# Nettoyage du jeu de données Titanic

## 1. Contexte et objectif

Nous appliquons ici le nettoyage « fixe » du jeu d'entraînement : des règles qui ne dépendent pas des statistiques des données et qui peuvent donc être appliquées avant la séparation train/test. Les imputations d'`Age` et `Embarked`, la mise à l'échelle et l'encodage seront réalisés plus tard dans un pipeline, calculés sur le train uniquement pour éviter toute fuite de données.

## 2. Import et chargement des données

Nous importons pandas et chargeons le jeu brut. Nous travaillons ensuite sur une copie afin de conserver le jeu brut intact.

In [1]:
import pandas as pd

df = pd.read_csv("../data/dataset-titanic-6afac2fa9ddd8043393546.csv")
df_propre = df.copy()  # Nous travaillons sur une copie pour garder le jeu brut intact
df_propre.shape

(891, 12)

Nous constatons que le jeu brut compte 891 lignes et 12 colonnes. Nous le copions dans `df_propre` pour toutes les transformations suivantes.

## 3. Nettoyage des noms

Nous comptons d'abord les noms comportant des espaces en trop, puis nous les supprimons avec `str.strip()`.

In [2]:
# Nous comptons les noms avec des espaces en trop avant correction
nb_espaces = (df_propre["Name"] != df_propre["Name"].str.strip()).sum()
print(nb_espaces)

# Nous supprimons les espaces en début et en fin de chaîne
df_propre["Name"] = df_propre["Name"].str.strip()

2


Nous constatons 2 noms avec des espaces en trop, que nous avons corrigés. Le texte est désormais propre pour l'extraction du titre.

## 4. Extraction du titre

Le titre (Mr, Mrs, Miss, Master…) est situé entre la virgule et le premier point du nom. Nous l'extrayons avec une expression régulière, car il peut apporter une information utile sur le profil du passager.

In [3]:
df_propre["Titre"] = df_propre["Name"].str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
df_propre["Titre"].value_counts()

Titre
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

Nous constatons 17 titres différents, dont quelques-uns très rares (Lady, Sir, Don, Capt…). Les titres fréquents dominent largement.

## 5. Regroupement des titres rares

Nous harmonisons les variantes françaises (Mlle, Ms → Miss ; Mme → Mrs), puis nous regroupons tous les titres autres que Mr, Mrs, Miss et Master dans une catégorie « Autre » pour éviter des catégories trop petites.

In [4]:
df_propre["Titre"] = df_propre["Titre"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
titres_frequents = ["Mr", "Mrs", "Miss", "Master"]
df_propre["Titre"] = df_propre["Titre"].where(df_propre["Titre"].isin(titres_frequents), "Autre")
df_propre["Titre"].value_counts()

Titre
Mr        517
Miss      185
Mrs       126
Master     40
Autre      23
Name: count, dtype: int64

Nous obtenons 5 catégories : Mr (517), Miss (185), Mrs (126), Master (40) et Autre (23).

## 6. Cabine renseignée ou non

`Cabin` compte 687 valeurs manquantes sur 891. Nous conservons l'information « cabine renseignée ou non » dans une variable binaire, puis nous vérifions qu'elle apporte quelque chose, y compris à classe égale.

In [5]:
df_propre["Cabine_Renseignee"] = df_propre["Cabin"].notna().astype(int)

# Lien avec la classe : la variable est-elle redondante avec Pclass ?
print(pd.crosstab(df_propre["Pclass"], df_propre["Cabine_Renseignee"]))
print(df_propre["Cabine_Renseignee"].corr(df_propre["Pclass"]).round(3))

# Taux de survie selon l'indicateur, puis à classe égale
print(df_propre.groupby("Cabine_Renseignee")["Survived"].mean().round(3))
print(df_propre.groupby(["Pclass", "Cabine_Renseignee"])["Survived"].mean().round(3))

Cabine_Renseignee    0    1
Pclass                     
1                   40  176
2                  168   16
3                  479   12
-0.726
Cabine_Renseignee
0    0.300
1    0.667
Name: Survived, dtype: float64
Pclass  Cabine_Renseignee
1       0                    0.475
        1                    0.665
2       0                    0.440
        1                    0.812
3       0                    0.236
        1                    0.500
Name: Survived, dtype: float64


Nous constatons que l'indicateur est fortement lié à `Pclass` (corrélation de -0,73) : la quasi-totalité des cabines renseignées concernent la 1ʳᵉ classe (176 passagers sur 216). Il garde néanmoins un effet propre : le taux de survie est de 0,667 avec une cabine renseignée contre 0,300 sans, et l'écart persiste à classe égale (1ʳᵉ classe : 0,665 contre 0,475 ; 2ᵉ : 0,812 contre 0,440 ; 3ᵉ : 0,500 contre 0,236). Nous conservons donc cet indicateur, avec la prudence habituelle : une corrélation n'est pas une causalité.

## 7. Suppression des colonnes devenues inutiles

Nous retirons `Cabin` (remplacée par l'indicateur), `Name` (le titre en a été extrait) et `Ticket`, un texte libre qui compte 681 valeurs différentes pour 891 lignes. Nous conservons `PassengerId`, utile pour la soumission, mais nous l'exclurons des variables explicatives au moment de la modélisation.

In [6]:
df_propre = df_propre.drop(columns=["Cabin", "Name", "Ticket"])
df_propre.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PassengerId        891 non-null    int64  
 1   Survived           891 non-null    int64  
 2   Pclass             891 non-null    int64  
 3   Sex                891 non-null    str    
 4   Age                714 non-null    float64
 5   SibSp              891 non-null    int64  
 6   Parch              891 non-null    int64  
 7   Fare               891 non-null    float64
 8   Embarked           889 non-null    str    
 9   Titre              891 non-null    str    
 10  Cabine_Renseignee  891 non-null    int64  
dtypes: float64(2), int64(6), str(3)
memory usage: 76.7 KB


Nous constatons qu'il reste 11 colonnes : `PassengerId`, `Survived`, `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`, `Titre` et `Cabine_Renseignee`. Il reste volontairement 177 valeurs manquantes pour `Age` et 2 pour `Embarked` : nous les imputerons dans le pipeline, sur le train uniquement.

## 8. Sauvegarde du jeu nettoyé

Nous enregistrons le jeu nettoyé dans le dossier `data/`, qui est exclu de Git, pour l'utiliser dans le notebook de modélisation.

In [7]:
df_propre.to_csv("../data/titanic_nettoye.csv", index=False)

## 9. Conclusion

Nous avons nettoyé les noms, extrait et regroupé les titres, créé l'indicateur de cabine renseignée et retiré les colonnes inutiles. Le jeu nettoyé est prêt pour l'étape suivante : séparation train/test, pipeline de prétraitement (imputation, échelle, encodage) et entraînement des trois modèles.